# Characterize graph hierarchy

Use the nested-SBM descriptors on PyTorch Geometric graphs. Install the optional `graph-tool` dependency as described in the [README](../README.md#installation).

In [ ]:
from pathlib import Path
import sys
import tempfile

if not (Path.cwd() / "src").is_dir():
    sys.path.insert(0, str(Path.cwd().parent))

import networkx as nx
from torch_geometric.utils import from_networkx

from src.datasets import SyntheticHierarchicalData, generate_synthetic_dataset
from src.hierarchy import compute_hierarchy_stats, NestedSBMStats

## Score a graph

Compare a graph with planted hierarchical communities to a random graph with the same number of nodes and edges. Inference is stochastic, and finite graphs need not show a clear separation.

In [ ]:
sizes = [12] * 8
probabilities = [[0.65 if i == j else (0.08 if i // 4 == j // 4 else 0.003)
                  for j in range(8)] for i in range(8)]
planted = nx.stochastic_block_model(sizes, probabilities, seed=42)
null = nx.gnm_random_graph(planted.number_of_nodes(), planted.number_of_edges(), seed=42)

for name, graph in [("planted", planted), ("null", null)]:
    stats = compute_hierarchy_stats(from_networkx(graph), seed=42)
    print(name, "count:", stats["substructure_count"], "depth:", stats["substructure_depth"])

`substructure_count` counts blocks at the finest inferred level. `substructure_depth` follows the paper's analysis script and counts stored levels, including any repeated single-block root levels.

## Score a dataset

Pass `NestedSBMStats` as a `pre_transform` to cache the descriptors. Replace the example path with a dataset directory containing `raw/graph_N.pt` files.

In [ ]:
# Create a small local example; use your own dataset path instead.
example_root = Path(tempfile.mkdtemp(prefix="psa-hierarchy-")) / "macro"
generate_synthetic_dataset(
    example_root, name="macro", num_graphs=1, seed=42,
    target_num_nodes=64, M_choices=(4,), K_choices=(2,),
)

dataset = SyntheticHierarchicalData(
    example_root,  # e.g. "/path/to/your/dataset"
    pre_transform=NestedSBMStats(seed=42),
)
graph = dataset[0]
print("count:", graph.substructure_count.item(), "depth:", graph.substructure_depth.item())